# ASoVi Imager — フルパイプライン (headless)

WFCI 解析パイプラインを **Dear PyGui ダッシュボードと同じ順序・同じ処理**で、
notebook から Stage ごとに実行するためのファイルです。

Stage 構成 (`runner.PipelineSession` / GUI の Stages 表と同一):

| # | Stage | 内容 |
|---|-------|------|
| 1 | **preprocess** | DFT registration + WFCI 線形減算 (dF/F) + binning |
| 2 | **pca** | annotation source (dF/F) の PCA |
| 3 | **ica** | 空間 ICA デノイズ (`skip_ica=True` で無効化) |
| 4 | **annotation** | Allen atlas への control-point warp |
| 5 | **roi** | ROI 時系列の抽出 |
| 6 | **correlation** | ROI 相関ネットワーク |
| 7 | **export** | warp 済み dF/F の movie / tiff / mat 書き出し |

## 0. セットアップ

リポジトリのルート（`src/` の 1 つ上）で notebook を起動しているか、
下のセルで自動的に `sys.path` を通します。

In [ ]:
import sys
from pathlib import Path

# --- src/ を sys.path に追加 (未インストールの checkout でも動くように) ---
_here = Path.cwd()
for _root in (_here, *_here.parents):
    if (_root / "pyproject.toml").is_file() and (_root / "src" / "asvimg").is_dir():
        if str(_root / "src") not in sys.path:
            sys.path.insert(0, str(_root / "src"))
        REPO_ROOT = _root
        break
else:
    raise RuntimeError("repo root (src/asvimg を含む階層) が見つかりません")

print("REPO_ROOT =", REPO_ROOT)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display

from asvimg import PipelineConfig, save_config, load_config
from asvimg.runner import PipelineSession, completed_stages, stage_artifacts


class NotebookReporter:
    """PipelineSession のイベントを notebook に表示する reporter。

    - on_stage    : ステージの開始/終了を 1 行で表示
    - on_log      : ログ行をそのまま print
    - on_progress : 末尾を上書きせず、区切りのいいところだけ表示
    - on_figure   : matplotlib figure を inline 表示 (表示後に close)
    """

    def __init__(self, show_progress=False):
        self.show_progress = show_progress

    def on_stage(self, stage, status, *, message="", elapsed=None):
        tail = f"  ({elapsed:.1f}s)" if elapsed else ""
        extra = f" — {message}" if message else ""
        print(f"[{stage}] {status}{tail}{extra}")

    def on_progress(self, stage, current, total, *, message=""):
        if self.show_progress and total and (current == total or current % max(1, total // 5) == 0):
            print(f"    {stage}: {current}/{total} {message}")

    def on_log(self, stage, text, *, level="info"):
        print(text)

    def on_figure(self, stage, key, figure):
        display(figure)
        plt.close(figure)


print("ready")

## 1. 設定 (Configuration)

GUI の Configuration ペインと同じ `PipelineConfig` を使います。
下のセルは **db / ops のパラメータを一通り並べた dict** です（教示用に
デフォルト値も隠さず書いています。フィールドの完全な一覧は
`python -c "import dataclasses as d; from asvimg import PipelineConfig as C; [print(f.name) for f in d.fields(C)]"`）。

> ⚠️ `input_dir` に入っている `_sampleData04_hcimg` は **配布物には含まれません**
> （`.gitignore` 対象）。自分のデータのパスに書き換えてください。

- **db** … 再現性に関わる Data セクション (入出力先・形式・実験名 …)
- **ops** … それ以外の全処理パラメータ (Channels / Preprocess / Annotation /
  PCA·ICA / ROI / Outputs / Movie / Atlas)

各行のコメントが GUI のツールチップ相当の説明です。値を書き換えれば
そのまま反映されます。

In [ ]:
# ============================================================================
#  db + ops パラメータ (教示用にデフォルトも明示)。完全な一覧は
#  dataclasses.fields(PipelineConfig) を参照 (上のセルのコマンド)。
#  値は _sampleData04_hcimg（2ch: source / donner）向け — 自分のデータに変更のこと。
# ============================================================================
PARAMS = dict(
    # ---- db (Data / 再現性) -------------------------------------------------
    input_dir=str(REPO_ROOT / "Analysis" / "_sampleData04_hcimg"),  # 生データ (dcimg/tif/sifx) のディレクトリ (配布物には無い)
    output_dir=None,                # 出力先。None -> input_dir/asi/<output_format>
    output_format="npy",            # 保存形式: "mat" | "npy" | "h5"
    exp_name="rysk",                # 実験名 (出力ファイル名の stem)。None -> ファイル名から推定
    output_metadata_yaml=True,      # 入力メタデータ YAML を保存するか
    dcimg_backend="auto",           # dcimg 読込: "auto" | "sdk" | "native"

    # ---- Channels -----------------------------------------------------------
    channels_name=["GCaMP", "GCaMP"],     # 取得サイクルのチャンネル名 (この順で読む)
    channels_prop=["source", "donner"],   # 各chの役割: "source"(蛍光) / "donner"(補正用)。"s"/"d" 可
    channels_slip=None,             # ファイル毎の位相ズレ (読み込み順に1つずつ)。None -> なし
    ch_for_annotation=0,            # cpselect が使う group を選ぶ ch (0-based)。分解は全 group で走る
    template_ch=0,                  # registration テンプレート作成に使うサイクル ch (0-based)
    demux_qc=True,                  # 強度フィンガープリントで demux 位相を QC (検出のみ)
    demux_start_offset=0,           # demux 位相の全体回転 (0..cycle_len-1)。frame0 = channels_name[この値]

    # ---- Preprocess ---------------------------------------------------------
    do_registration=True,           # DFT registration (動き補正) を行うか
    registration_cache="cached",    # "force": 毎回再実行 / "cached": *完了した* 前回 run (reg_meta.npz あり & 非partial) があればスキップ
    registration_batch_size=500,    # registration のバッチ (read->register->bin->write) フレーム数
    usfac=50,                       # DFT サブピクセル精度 1/usfac px (50 が速度/精度の実用点)
    use_mmap=False,                 # dF/F 読み戦略: False=全ch RAM(短時間/大メモリ) / True=memmap+row-strip(低メモリ)。出力同一
    linear_subt=True,               # WFCI 線形減算 (dF/F 補正) を行うか。False だと dff_*.npy を一切書かない
    baseline_percentile=5.0,        # dF/F baseline に使うピクセル毎パーセンタイル (0-100)
    detrend=False,                  # 回帰前の指数(退色)デトレンド (MATLAB flag_ExpoSub)
    baseline_percentile_highpass=False,     # 回帰前の rolling-percentile ハイパス (MATLAB flag_BaselineFilter)
    baseline_percentile_highpass_sec=120.0, # そのウィンドウ長 (秒)
    baseline_percentile_highpass_rank=50.0, # そのパーセンタイル順位 (50 = 中央値)
    hemovar_qc=True,                # 血流回帰の分散説明率 (R^2) マップを保存 (donner のある group のみ)
    start_initial_frames=None,      # 回帰/baseline/ROI図/相関から除く先頭フレーム数。None -> 自動(4*fps/cycle_len)
    ignore_last_frames=0,           # 回帰/ROI図/相関から除く末尾フレーム数 (0 -> 全部使う)
    binning=2,                      # 空間ビニング係数 (2 -> 2x2, 0 -> なし)
    flip=False,                     # 左右反転 (テンプレートも一緒に反転される)
    fps=40,                         # カメラ fps (全ch合算)
    template_stride=100,            # テンプレート平均に使うサイクル間隔
    template=None,                  # テンプレート画像パス。None -> 自動生成
    filter_xyt=None,                # reg後の3Dフィルタ [x,y,t] 例:[1,1,1] (全編連続適用)。None -> なし
    filter_xyt_kind="mean",         # filter_xyt の種類: "mean" | "median" | "gaussian"
    max_frames=None,                # フレーム数上限 (動作確認用)。None -> 全部
    delete=True,                    # run 前に既存出力を削除するか (marks.mat / ica_exclusion.json / rois.csv / demux_correction.json は残す)

    # ---- Annotation (atlas warp) --------------------------------------------
    annotation="gui",             # "cache": 保存済 marks.mat / "gui": cpselect で選択 / False: skip
    annotation_allow_reflection=False,  # atlas 変換に鏡映(左右反転)を許すか (非中線の制御点が必要)
    annotation_atlas_path=str(REPO_ROOT / "Analysis" / "matlab" / "wfciAnnotationData.mat"),  # Allen atlas .mat
    post_annotation_time_average=1, # warp後の時間移動平均の半窓 N (0:off, 1: ±1=3フレーム平均)。ROI信号には効かない
    post_annotation_filter_xyt=None,    # warp後の 3D フィルタ [x,y,t]。None -> なし (ROI信号には効かない)
    post_annotation_filter_kind="mean", # その種類: "mean" | "median" | "gaussian"

    # ---- PCA / ICA ----------------------------------------------------------
    pca_n_components=10,            # PCA 成分数
    pca_smooth_sigma=5.0,           # PCA 成分の時間平滑化 Gaussian sigma (表示専用; 成分番号は不変)
    pca_skip_frames=10,            # PCA/ICA の fit 入力の時間間引き (1=全フレーム, fit専用; 適用は全長)
    skip_ica=True,                  # ICA デノイズを完全にスキップするか (ica_denoise="subtract" とは併用不可)
    ica_n_components=None,          # ICA 成分数。None -> PCA と同数
    ica_max_iter=1000,             # ICA 最大反復数
    ica_random_state=0,            # ICA 乱数シード
    ica_exclusion="cache",          # 除外IC: "cache"(ica_exclusion.json を再生) | "gui" | {"GCaMP":[2,7]}(0-based) | False
    ica_denoise="off",              # "off": 除外は QC のみ / "subtract": 下流が ica/dff_{name}.npy を読む

    # ---- ROI / correlation --------------------------------------------------
    roi_signal="dff",               # 抽出する ROI 信号: "Both" | "Raw" | "dff"
    save_roi_signals="both",        # ROI 時系列の書き出し: None | "csv" | "pickle" | "both"
    corr_method="gsr",              # ROI 相関: "raw"(素Pearson) / "gsr"(全体信号回帰後) / "partial"(偏相関)
    corr_auto_threshold=True,       # エッジ閾値: 自動(密度ベース) か 固定 corr_network_threshold か
    corr_edge_density=0.25,         # 自動時: 残す強いエッジの割合 (0.25 = 上位25%)
    corr_network_threshold=0.30,    # 固定閾値 (corr_auto_threshold=False の時の |r| カットオフ)

    # ---- Outputs: Frames ----------------------------------------------------
    save_raw_each_ch=False,         # 生 (前処理前) channels を TIFF 保存
    save_registered_each_ch=False,  # registered+binned channels を TIFF 保存
    save_annotated_each_ch=True,    # atlas-warp 済 channels を TIFF 保存
    save_annotated_dF_mat=True,     # atlas-warp 済 dF/F を group毎に mat/npy/h5 保存
    export_orientation="HWT",       # 最終export(dfWarped)の軸順: "HWT"(H,W,T; MATLAB/旧互換) | "THW"
    tiff_format="big-tiff",         # "big-tiff" | "ome-tiff"
    tiff_compression=True,          # TIFF の zlib 圧縮
    # ---- Outputs: Movie -----------------------------------------------------
    save_movie=True,                # atlas-warp 済 dF/F の動画を保存
    save_movie_speed=2.0,           # 実時間 ×N (出力fps = fps_ch * speed)
    save_movie_codec="MJPG",        # FourCC codec (cv2.VideoWriter)
    save_movie_vminmax=(-2.0, 10.0),  # dF/F 表示レンジ (vmin, vmax) [%]
    save_movie_cmap="magma",        # 動画の LUT: "magma" | "turbo" | "gray" | "viridis"
    # ---- Outputs: Figures ---------------------------------------------------
    save_figures="png+pdf",         # QC図の保存: "none" | "png" | "png+pdf"
)

config = PipelineConfig(**PARAMS)

# 出力先を解決 (output_dir=None -> input_dir/asi/<format>)
from asvimg.config import default_output_dir
OUTPUT_DIR = (Path(config.output_dir) if config.output_dir
              else default_output_dir(Path(config.input_dir), config.output_format))

print("input_dir  =", config.input_dir)
print("output_dir =", OUTPUT_DIR)
print("channels   =", list(zip(config.channels_name, config.channels_prop)))
print("skip_ica   =", config.skip_ica, "| annotation =", config.annotation,
      "| corr_method =", config.corr_method)

### セッションの作成

`PipelineSession` は完全にヘッドレスです。annotation の対話選択が必要な
モード (`annotation="gui"`) のときだけ、`GuiBridge` 経由で **別プロセスの
cpselect** が起動します（notebook カーネルには dpg context を作りません）。
`annotation="cache"`（デフォルト）なら保存済み `marks.mat` を読み込みます。

In [ ]:
from asvimg.gui.bridge import GuiBridge

reporter = NotebookReporter(show_progress=True)

# GuiBridge の provider は:
#   annotation=="gui"  -> 別プロセスで cpselect を起動 (安全)
#   それ以外           -> resolve_marks で非対話に解決 (cache/coords/False)
bridge = GuiBridge()

session = PipelineSession(
    config,
    reporter=reporter,
    marks_provider=bridge.marks_provider,
    ica_provider=bridge.ica_provider,
    cancel=bridge.cancel,
)
print("session ready — output_dir:", session.state.output_dir)

# 既存の成果物から、どの Stage が完了済みか確認 (GUI の Stages 表相当)
for stage, done in completed_stages(session.state.output_dir).items():
    print(f"  {stage:12s} {'done' if done else 'pending'}")

---
以下、各 Stage を順に実行します。各 `run_*()` は独立して再実行でき、
必要な上流の成果物はディスクから読み直されます（GUI の per-stage Run と同じ）。

## Stage 1 — preprocess

DFT registration (`usfac`) で揺れ補正 → binning → WFCI 線形減算 (dF/F)。
登録済みは ch 毎の連続 memmap `reg_Ch{i}.npy` (T,H,W)、dF/F は全編一括 (global baseline) の
`dff_{name}.npy` に出力します。`registration_cache="cached"` のときは、**完了した前回の run**
（`reg_meta.npz` があり `partial` でない）がある場合だけ registration をスキップします
（中断・クラッシュした run の中途半端な `reg_Ch*.npy` は再実行されます）。

In [ ]:
stats = session.run_preprocess()
stats

## Stage 2 — PCA

annotation source チャンネルの dF/F に対して PCA。空間マップ (PC*) を出力し、
`save_figures` が有効なら QC 図を保存します。

In [ ]:
pca_result = session.run_pca()
pca_result

## Stage 3 — ICA (デノイズ)

空間 ICA によるデノイズ。`config.skip_ica=True` の場合、このステージは
**no-op**（SKIPPED）になります。対話的に除外 IC を選ぶ場合は
`ica_provider`（`GuiBridge` 経由の別プロセス GUI）が使われます。

In [ ]:
excluded_ics = session.run_ica()
print("excluded ICs:", excluded_ics)

## Stage 4 — annotation (atlas warp)

Allen atlas (ACCFv3) への control-point warp を計算します。

- `annotation="cache"` (デフォルト) … 保存済み `marks.mat` を読み込み
- `annotation="gui"` … **別プロセスの cpselect** を起動して点を選択
  （notebook から呼んでも安全。ウインドウを閉じると結果が返ります）
- `annotation=False` … スキップ

> control-point をこの notebook から選び直したい場合は、上の Config セルで
> `config = ... annotation="gui"` として `session` を作り直してから実行してください。

In [ ]:
tform = session.run_annotation()
tform

## Stage 5 — ROI 抽出

ROI 時系列を抽出します。カスタム ROI があれば `rois.csv`、無ければ atlas の
デフォルト ROI を使います。`save_roi_signals`（csv / pickle / both）に従って
時系列を書き出します。

In [ ]:
roi_out = session.run_roi()
list(roi_out.keys()) if isinstance(roi_out, dict) else roi_out

## Stage 6 — correlation ネットワーク

ROI 間相関を計算し、ネットワーク図を描きます。`corr_method`:

- `raw` … 素の Pearson（グローバル信号で 1 に張り付きがち）
- `gsr` … global-signal regression 後
- `partial` … 偏相関

エッジは `corr_auto_threshold`（密度ベース）または固定 `corr_network_threshold`。

In [ ]:
corr_out = session.run_correlation()
list(corr_out.keys()) if isinstance(corr_out, dict) else corr_out

## Stage 7 — export

atlas-warp 済み dF/F を movie / tiff / mat などに書き出します
（`save_movie`, `save_annotated_*`, `save_annotated_dF_mat` などの Outputs 設定に従う）。

In [ ]:
session.run_export()
print("export done")

---
## まとめて実行 / 成果物の確認

全ステージを一括で回すなら `run_all()`。上で個別に実行済みなら不要です。

In [ ]:
# state = session.run_all()   # 一括実行したい場合はコメントアウトを外す

# 出力された成果物の一覧
for stage, path in stage_artifacts(session.state.output_dir).items():
    print(f"  {stage:12s} {path}")

### 設定の保存

実行に使った設定を `db.yaml` + `ops.yaml` として出力先に保存できます
（GUI の Save config 相当）。

In [ ]:
save_config(config, session.state.output_dir)
print("saved db.yaml + ops.yaml to", session.state.output_dir)

---
### 補足: ROI の編集について

ROI の追加・削除・対側コピー・atlas/warp オーバーレイ表示は、
**ダッシュボード GUI の ROI エディタ**（in-process dpg）で行います。
notebook では dpg context を作らない方針のため、ROI を変更する場合は:

1. ダッシュボード (`python -m asvimg.gui`) の **Edit ROIs** で編集する、または
2. 出力ディレクトリの **`rois.csv`** を直接編集する

いずれかの後、このノートの **Stage 5 (roi)** 以降を再実行してください。